## Notes

The first class is half philosophical and half technical.

The philosophical part introduced the Bayesian (scientific research) workflow. Statistics is about correlation, all causality arguments came from the domain theory. 



The practical part did a Bayesian update on a Bernoulli distributed variable, using the "forking path of data" method.
We started with a percentage of water on the planet experiment, and collected some data, and showed how the posterior updates with the stream of data. However it's not clear *how* the update happened, just the plots. To actually understand how it works we have to work with the Beta prior and the Bernoulli likelihood? These are for later lectures.

Then we jumped to a second, much simpler experiment. Still percentage of water on the planet, but an equal prior, with only 5 possible values, 0, 1, 2, 3, 4 out of 4 portion are water, and with a smaller data set of 3 obervations, WLW. However it's also shown that more data, or more elaborate prior can be easily incorporated into the calculation, since the Bayesian update process is multiplicative.

This is the grid approximation of the water percentage problem, we have discretized a continuous variable between 0 and 1 to a discrete one with only 5 possible values.

we can count the number of ways that an observed sequence of data can happen, by sequentially multiplying new counts with existing ones.

In [6]:
portions_water = [0, 1, 2, 3, 4]
portions_land = [4 - pw for pw in portions_water]
obs = [1, 0, 1]

counts = [1, 1, 1, 1, 1] # all proposals are equally likely before observations
for ob in obs:
    if ob == 1:
        for i, (c, p) in enumerate(zip(counts, portions_water)):
            counts[i] = c*p
    if ob == 0:
        for i, (c, p) in enumerate(zip(counts, portions_land)):
            counts[i] = c*p

counts # final counts, after observing the data

[0, 3, 8, 9, 0]

the counts are then normalized to probabilities

In [7]:
probs = [c/sum(counts) for c in counts]
probs

[0.0, 0.15, 0.4, 0.45, 0.0]

we can also work directly with proportions, and the likelihood function for Bernoulli data

In [8]:
import jax.numpy as jnp

props = jnp.array([0., 0.25, 0.5, 0.75, 1.])
posterior = jnp.ones(5)  # uniform prior
for ob in obs: # streaming observations, online update
    likelihood = props**ob * (1-props)**(1-ob)
    posterior = posterior * likelihood

posterior/sum(posterior)

Array([0.  , 0.15, 0.4 , 0.45, 0.  ], dtype=float32)

## homework

In the homework problem, 10 participants, flipped the same coint, reported head or tail, and if reported head win a 10 euros cash prize. As it turned out 8 of them claimed the prize, and we want to know 
1. how many ways the observed data can be realized, if all participants are honest;
1. how many ways the observed data can be realized, if only 5 of the participants are honest;
1. the number of honest participants that maximize the number of ways the observed data can be realized.


so this is an inference problem about the number (or portion) of honest participants.
However, note that this is not what the problem is asking. It's an "if" question: if there are `h` honest participant, how many ways the data can be realized. So simply choosing a certain number in the 10, a combinatorial problem.

we have 10 participants, 8 reported heads; so the other 2 must be honest, and the possible number of honest participants are [2, 10].

If all 10 participants are honest, since each coin toss is half-half, we have 2 participants tossed tails while the other eight tossed heads. Choosing 2 tails in 10 tosses, the possible combo is `comb(10, 2) * comb(8, 8)`. 
And the probability of observing such an event is `comb(10, 2) * (1/2)**2 * comb(8, 8) * (1/2)**8`.

If only 5 participants are honest and we observed 8 heads, 2 of the 5 honest tossed tails, 3 tossed heads, and the 5 dishonest participants all *reported* heads. the possible combo is `comb(5, 2) * comb(3, 3) * comb(5, 5)`.
And the probability of observing such an event is `comb(5, 2) * (1/2)**2 * comb(3, 3) * (1/2)**3 * comb(5, 5) * 1**5`.

More generally, if there are `h` honest participants, then 2 tossed tails, h-2 tossed heads, the nother 10-h *reported* heads.
the possible combo is `comb(h, 2) * comb(h-2, h-2) * comb(10-h, 10-h) = comb(h, 2)`.
And the probability of observing such an event is `comb(h, 2) * (1/2)**2 * comb(h-2, h-2) * (1/2)**(h-2) * comb(10-h, 10-h) * 1**(10-h) = comb(h, 2) * (1/2)**h`.

In [9]:
from math import comb

likelihood = []
combos = []
N = 10
for h in range(2, N+1):
    c = comb(h, 2)
    p = c * 0.5**h
    combos.append(c)
    likelihood.append(p)
    print(h, c, p)



2 1 0.25
3 3 0.375
4 6 0.375
5 10 0.3125
6 15 0.234375
7 21 0.1640625
8 28 0.109375
9 36 0.0703125
10 45 0.0439453125


to get the posterior for `h` we have to normalize the likelihood (assuming a flat prior)

In [10]:
posterior = [l/sum(likelihood) for l in likelihood]
posterior


[0.1292276627965674,
 0.19384149419485108,
 0.19384149419485108,
 0.16153457849570924,
 0.12115093387178193,
 0.08480565371024736,
 0.05653710247349823,
 0.03634528016153458,
 0.02271580010095911]

One step further. If we are to build a generative story:
1. the coin fairness unknow, so p~Beta
2. the coin toss outcome xn~Bernoulli(p)
3. the honesty of each participant hn~Beta
4. the outcome should be a mixture distribution, 1 if the coin toss outcome is 1 or participant not honest, 0 if toss outcome is 0 AND participant honest.

